# 🚀 Distributed Semiconductor Image Restoration on Kaggle (T4 x2 Dual-GPU)

This notebook trains **NAFNet-SR** using **Dual NVIDIA Tesla T4 GPUs (T4 x2)** with **PyTorch DataParallel**, **In-Memory RAM Streaming**, **Automatic Mixed Precision (AMP FP16)**, **Model EMA**, and **Calibrated Composite Metrology Loss**.


## Step 1: Verify Dual GPU (Tesla T4 x2)


In [ ]:
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
print('GPU Count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  [+] GPU {i}: {torch.cuda.get_device_name(i)}')


## Step 2: Clone Repository (Branch `Kunal`)


In [ ]:
!git clone -b Kunal https://github.com/kmbeddedd/semicon_2026.git
%cd semicon_2026


## Step 3: Install Dependencies


In [ ]:
!pip install -q -r requirements.txt


## Step 4: Extract Dataset & Setup Data Directory

Extracts the canonical `train.zip` and `Test_NoisyLR.zip` archives directly into `data/`.

In [ ]:
import os, glob, shutil, random, zipfile

os.makedirs('data', exist_ok=True)

# 1. Extract the two canonical repository archives
bundled_archives = [('train.zip', 'data'), ('Test_NoisyLR.zip', 'data/test')]
for z, destination in bundled_archives:
    if os.path.exists(z):
        print(f'  [+] Extracting {z} into {destination}...')
        os.makedirs(destination, exist_ok=True)
        with zipfile.ZipFile(z, 'r') as zip_ref:
            zip_ref.extractall(destination)

# 2. If additional zips exist in /kaggle/input/, extract them
for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
    print(f'  [+] Extracting /kaggle/input archive: {z}...')
    with zipfile.ZipFile(z, 'r') as zip_ref:
        zip_ref.extractall('data')

# 3. Create 10% validation split if not already present
if os.path.exists('data/train/NoisyLR') and not os.path.exists('data/val'):
    random.seed(42)
    os.makedirs('data/val/NoisyLR', exist_ok=True)
    os.makedirs('data/val/GT', exist_ok=True)
    exts = ('*.npy', '*.NPY', '*.png', '*.PNG', '*.jpg', '*.JPG')
    train_files = []
    for ext in exts:
        train_files.extend(glob.glob(f'data/train/NoisyLR/{ext}'))
    train_files = sorted(list(set(train_files)))
    train_files = [f for f in train_files if os.path.exists(os.path.join('data/train/GT', os.path.basename(f)))]
    if len(train_files) > 0:
        val_k = min(len(train_files), max(1, int(len(train_files) * 0.1)))
        val_samples = random.sample(train_files, k=val_k)
        for vf in val_samples:
            fn = os.path.basename(vf)
            shutil.move(vf, os.path.join('data/val/NoisyLR', fn))
            gt_p = os.path.join('data/train/GT', fn)
            shutil.move(gt_p, os.path.join('data/val/GT', fn))
        print(f'[+] Created 10% validation split ({len(val_samples)} samples in data/val/)')

print('\n=== Final Dataset Status ===')
print('Train NoisyLR count:', len(glob.glob('data/train/NoisyLR/*.*')) if os.path.exists('data/train/NoisyLR') else 0)
print('Train GT count:     ', len(glob.glob('data/train/GT/*.*')) if os.path.exists('data/train/GT') else 0)
print('Val NoisyLR count:  ', len(glob.glob('data/val/NoisyLR/*.*')) if os.path.exists('data/val/NoisyLR') else 0)
print('Val GT count:       ', len(glob.glob('data/val/GT/*.*')) if os.path.exists('data/val/GT') else 0)


## Step 5: Train NAFNet-SR on Dual T4 GPUs (Continuous High-Utilization Training)


In [ ]:
# Continue the accepted research checkpoint; retain it unless validation PSNR improves.
!mkdir -p /kaggle/working/weights
!cp weights/best_model.pt /kaggle/working/weights/best_model.pt
!python train.py --init_weights weights/best_model.pt --spectral_mixer --uncertainty_head --w_nll 0.02 --nll_beta 0.5 --extension_lr_multiplier 5 --epochs 20 --batch_size 16 --lr 2e-5 --warmup_epochs 1 --scale 2 --patch_size 0 --num_workers 2 --no_cache --save_dir /kaggle/working/weights


## Step 6: Evaluate & Benchmark (Single Pass & 8-Fold TTA)


In [ ]:
# Fast production evaluation (< 14ms)
!python eval.py --input_dir data/val/NoisyLR --target_dir data/val/GT --output_dir /kaggle/working/val_restored --weights /kaggle/working/weights/best_model.pt --scale 2 --batch_size 16 --no_tta --check_clean_damage

# 8-Fold TTA evaluation
!python eval.py --input_dir data/val/NoisyLR --target_dir data/val/GT --output_dir /kaggle/working/val_restored_tta --weights /kaggle/working/weights/best_model.pt --scale 2 --batch_size 16
